In [1]:
import pandas as pd
import anndata as ad
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

In [2]:
cell_type_mapping_file="/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Mouse_Splicing_Foundation/Figures/LeafletFA_Supplemental_Table_1.xlsx" 
# read the excel file 
cell_type_mapping = pd.read_excel(cell_type_mapping_file)
cell_type_mapping.head()

,cell_id,cell_name,cell_ontology_class,broad_cell_type,tissue,tissue_celltype,dataset
0,A10_B000120,A10_B000120,basal epithelial cell of tracheobronchial tree,Epithelial,Trachea,Trachea_Epithelial,TMS
1,A10_B000126,A10_B000126,bulge keratinocyte,Epithelial,Skin,Skin_Keratinocyte,TMS
2,A10_B000127,A10_B000127,myeloid cell,Immune,SCAT,SCAT_Myeloid,TMS
3,A10_B000166,A10_B000166,basal cell,Epithelial,Mammary_Gland,Mammary_Epithelial,TMS
4,A10_B000169,A10_B000169,endothelial cell of coronary artery,Endothelial,Heart,Heart_VascEndo,TMS


In [3]:
BASE_DIR = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION"
SPLICE_INPUT = f"{BASE_DIR}/MODEL_INPUT/102025/model_ready_aligned_splicing_data_20251009_024406.h5ad"

# Read the splicing data
splice_adata = ad.read_h5ad(SPLICE_INPUT)
splice_adata

/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 142315 × 89831
    obs: 'cell_id_index', 'age', 'cell_ontology_class', 'mouse.id', 'sex', 'subtissue', 'tissue', 'dataset', 'cell_name', 'cell_id', 'cell_clean', 'specific_cell_type', 'broad_cell_type', 'medium_cell_type', 'seqtech', 'library_size', 'total_junction_reads', 'annotated_junction_reads', 'unannotated_junction_reads', 'n_detected_annotated_junctions', 'n_detected_unannotated_junctions'
    var: 'index', 'junction_id', 'event_id', 'splice_motif', 'annotation_status', 'gene_name', 'gene_id', 'num_junctions', 'position_off_5_prime', 'position_off_3_prime', 'CountJuncs', 'junction_id_index', 'n_cells_detected', 'confidence', 'aging_gene', 'aging_lifespan_effect', 'aging_longevity_influence'
    obsm: 'X_library_size'
    layers: 'cell_by_cluster_matrix', 'cell_by_junction_matrix'

In [4]:
# remove all the old colummns in splice_adata.obs other than cell_id and cell_name that are in cell_type_mapping
# find columns in cell_type_mapping that are in splice_adata.obs
columns_to_drop = [col for col in cell_type_mapping.columns if col in splice_adata.obs.columns] 
# remove cell_id and cell_name from columns_to_drop
columns_to_drop.remove("cell_id")
columns_to_drop.remove("cell_name")
print(columns_to_drop)

['cell_ontology_class', 'broad_cell_type', 'tissue', 'dataset']


In [5]:
# remove columns from splice_adata.obs
splice_adata.obs.drop(columns=columns_to_drop, inplace=True)
# reset index of splice_adata.obs
splice_adata.obs.reset_index(drop=True, inplace=True)
splice_adata.obs = pd.merge(splice_adata.obs, cell_type_mapping, on=["cell_id", "cell_name"], how="left")

# check that splice_adata.obs.index is exactly the same as splice_adata.obs["cell_id_index"]
assert np.all(splice_adata.obs.index == splice_adata.obs["cell_id_index"]), "Index of splice_adata.obs is not exactly the same as cell_id_index"
splice_adata.obs_names = splice_adata.obs["cell_id"]

In [6]:
splice_adata

AnnData object with n_obs × n_vars = 142315 × 89831
    obs: 'cell_id_index', 'age', 'mouse.id', 'sex', 'subtissue', 'cell_name', 'cell_id', 'cell_clean', 'specific_cell_type', 'medium_cell_type', 'seqtech', 'library_size', 'total_junction_reads', 'annotated_junction_reads', 'unannotated_junction_reads', 'n_detected_annotated_junctions', 'n_detected_unannotated_junctions', 'cell_ontology_class', 'broad_cell_type', 'tissue', 'tissue_celltype', 'dataset'
    var: 'index', 'junction_id', 'event_id', 'splice_motif', 'annotation_status', 'gene_name', 'gene_id', 'num_junctions', 'position_off_5_prime', 'position_off_3_prime', 'CountJuncs', 'junction_id_index', 'n_cells_detected', 'confidence', 'aging_gene', 'aging_lifespan_effect', 'aging_longevity_influence'
    obsm: 'X_library_size'
    layers: 'cell_by_cluster_matrix', 'cell_by_junction_matrix'

In [7]:
# Save the splicing data to BASE_DIR named Figure1_input_data.h5ad
splice_adata.write_h5ad(f"{BASE_DIR}/Figure1_input_data.h5ad")

In [8]:
print(f"Done saving the splicing data to {BASE_DIR}/Figure1_input_data.h5ad")

Done saving the splicing data to /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/Figure1_input_data.h5ad
